In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import numpy as np
import wandb

# Sieć neuronowa
class PolicyNetwork(nn.Module):
    def __init__(self, obs_dim, act_dim):
        super(PolicyNetwork, self).__init__()
        # Prosta sieć MLP, odpowiednia dla wektora obserwacji LunarLander (8 wartości)
        self.fc = nn.Sequential(
            nn.Linear(obs_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, act_dim),
            nn.Softmax(dim=-1) # Zwraca prawdopodobieństwa poszczególnych akcji
        )

    def forward(self, x):
        return self.fc(x)

#  Klasa algorytmu REINFORCE
class REINFORCE:
    def __init__(self, obs_dim, act_dim, lr=1e-3, gamma=0.99):
        self.policy = PolicyNetwork(obs_dim, act_dim)
        self.optimizer = optim.Adam(self.policy.parameters(), lr=lr)
        self.gamma = gamma
        
        # Pamięć epizodu
        self.saved_log_probs = []
        self.rewards = []

    def predict(self, state):
        """Wybiera akcję na podstawie stanu i zapisuje log-prawdopodobieństwo"""
        state = torch.from_numpy(state).float().unsqueeze(0)
        probs = self.policy(state)
        
        # Tworzy dystrybucję kategoryczną na podstawie prawdopodobieństw
        m = Categorical(probs)
        action = m.sample()
        
        # Zapisujemy log(pi(a|s)) do późniejszego obliczenia gradientu
        self.saved_log_probs.append(m.log_prob(action))
        
        return action.item()

    def update(self):
        """Aktualizuje wagi po zakończeniu epizodu"""
        R = 0
        returns = []
        
        # Obliczanie zdyskontowanych nagród G_t 
        for r in self.rewards[::-1]:
            R = r + self.gamma * R
            returns.insert(0, R)
            
        returns = torch.tensor(returns)
        
        # Normalizacja nagród. Bez tego REINFORCE w LunarLander może w ogóle nie zbiec.
        returns = (returns - returns.mean()) / (returns.std() + 1e-9)
        
        policy_loss = []
        for log_prob, Gt in zip(self.saved_log_probs, returns):
            # Ujemny znak, ponieważ PyTorch domyślnie robi minimalizację (gradient descent),
            # a my chcemy maksymalizować nagrodę (gradient ascent).
            policy_loss.append(-log_prob * Gt)
            
        self.optimizer.zero_grad()
        # Sumujemy straty z całego epizodu
        policy_loss = torch.cat(policy_loss).sum()
        policy_loss.backward()
        self.optimizer.step()
        
        # Czyszczenie pamięci po aktualizacji
        self.saved_log_probs = []
        self.rewards = []
        
        return policy_loss.item()

In [ ]:
def train_reinforce(env_name="LunarLander-v3", run_name="test_reinforce_2", max_episodes=2500, max_steps=1000):
    print(f"\n{'='*40}")
    print(f"Training REINFORCE")
    print(f"{'='*40}")

    run = wandb.init(
        project="nn2526_projekt3",
        name=run_name,
        config={"algo": "REINFORCE (Custom)", "env": env_name, "max_episodes": max_episodes},
        reinit=True
    )

    env = gym.make(env_name)
    
    obs_dim = env.observation_space.shape[0]
    act_dim = env.action_space.n
    
    agent = REINFORCE(obs_dim, act_dim, lr=5e-4, gamma=0.99)
    
    # Do śledzenia średniej nagrody
    running_reward = 0

    for episode in range(1, max_episodes + 1):
        state, _ = env.reset()
        ep_reward = 0
        
        for t in range(max_steps):
            action = agent.predict(state)
            state, reward, terminated, truncated, _ = env.step(action)
            
            agent.rewards.append(reward)
            ep_reward += reward
            
            if terminated or truncated:
                break
                
        # Aktualizacja wag po epizodzie
        loss = agent.update()
        
        # Wygładzona średnia nagroda do logowania
        running_reward = 0.05 * ep_reward + (1 - 0.05) * running_reward
        
        wandb.log({
            "episode": episode,
            "reward": ep_reward,
            "running_reward": running_reward,
            "loss": loss
        })
        
        if episode % 50 == 0:
            print(f"Episode {episode}\tLast Reward: {ep_reward:.2f}\tRunning Reward: {running_reward:.2f}")
            
        # Warunek wcześniejszego zakończenia: środowisko uważa się za rozwiązane przy średniej ~200
        if running_reward > 200:
            print(f"Solved at episode {episode}! Running reward is now {running_reward:.2f}")
            break

    # Zapisywanie modelu
    model_path = f"models/reinforce_{run_name}.pth"
    torch.save(agent.policy.state_dict(), model_path)
    
    env.close()
    run.finish()
    
    return model_path, agent

In [ ]:
import imageio
import os

def record_custom_agent(agent, filename="logs/videos/reinforce_lunarlander2.mp4", seconds=10, env_name="LunarLander-v3"):
    print(f"\n--- Recording {seconds}s video for Custom REINFORCE ---")

    # Upewniamy się, że folder docelowy istnieje
    os.makedirs(os.path.dirname(filename), exist_ok=True)

    # Tworzymy środowisko z trybem renderowania do obrazu
    env = gym.make(env_name, render_mode="rgb_array")

    # LunarLander działa zazwyczaj w 50 FPS
    fps = env.metadata.get("render_fps", 50)
    total_frames = fps * seconds

    frames = []
    state, _ = env.reset()

    import torch
    with torch.no_grad():
        for _ in range(total_frames):
            frames.append(env.render())
            
            state_tensor = torch.from_numpy(state).float().unsqueeze(0)
            probs = agent.policy(state_tensor)
            action = torch.argmax(probs, dim=-1).item() 
            
            state, reward, terminated, truncated, _ = env.step(action)

            if terminated or truncated:
                state, _ = env.reset()

    env.close()

    imageio.mimsave(filename, frames, fps=fps)
    print(f"Successfully saved video to: {filename}")

In [ ]:
if __name__ == "__main__":
    os.makedirs("models", exist_ok=True)
    os.makedirs("logs/videos", exist_ok=True)
    # trening
    model_path, trained_agent = train_reinforce(
        env_name="LunarLander-v3", 
        run_name="reinforce_run_2", 
        max_episodes=2500
    )

   
    record_custom_agent(
        agent=trained_agent, 
        filename="logs/videos/reinforce_flight.mp4", 
        seconds=15 
    )


Training REINFORCE


Episode 50	Last Reward: -296.94	Running Reward: -156.77
Episode 100	Last Reward: -600.30	Running Reward: -172.77
Episode 150	Last Reward: -316.71	Running Reward: -171.34
Episode 200	Last Reward: -99.01	Running Reward: -133.08
Episode 250	Last Reward: -467.09	Running Reward: -129.32
Episode 300	Last Reward: -145.96	Running Reward: -78.08
Episode 350	Last Reward: -141.98	Running Reward: -107.34
Episode 400	Last Reward: -25.98	Running Reward: -75.90
Episode 450	Last Reward: 40.92	Running Reward: -55.13
Episode 500	Last Reward: -20.44	Running Reward: -44.84
Episode 550	Last Reward: -8.80	Running Reward: -50.28
Episode 600	Last Reward: 26.40	Running Reward: -14.91
Episode 650	Last Reward: 11.85	Running Reward: -0.89
Episode 700	Last Reward: -5.84	Running Reward: -20.86
Episode 750	Last Reward: -57.81	Running Reward: -60.29
Episode 800	Last Reward: -23.13	Running Reward: -25.06
Episode 850	Last Reward: 9.44	Running Reward: 24.95
Episode 900	Last Reward: -6.33	Running Reward: 24.52
Episode 95

episode,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇██
loss,▂▃▃▂▂▃▂▃▂▂▃▂▃▂▁▃▃▃▃▂▃▃▄▃▄▃▄▂▃▃█▃▁▃▃▃▂▄▃▂
reward,▄▃▃▃▄▁▄▄▄▄▅▃▆▅▇▃▆▅▄▄▅▇▅▃▇▄▅▅▆▄█▅▆▆█▅█▆▄▆
running_reward,▁▂▃▃▃▃▃▃▃▄▅▅▄▄▄▅▃▄▆█▇▇▆▆▆▇▇▇▇▆▇████▇▇███
episode,2500
loss,-49.29293
reward,81.77468
running_reward,118.93736



--- Recording 15s video for Custom REINFORCE ---


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (600, 400) to (608, 400) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Successfully saved video to: logs/videos/reinforce_flight.mp4


In [ ]:
import torch
import gymnasium as gym

env_name = "LunarLander-v3"
env = gym.make(env_name)
obs_dim = env.observation_space.shape[0]
act_dim = env.action_space.n

loaded_agent = REINFORCE(obs_dim, act_dim)

model_path = "models/reinforce_reinforce_run_2.pth" 
loaded_agent.policy.load_state_dict(torch.load(model_path))
loaded_agent.policy.eval() 

record_custom_agent(
    agent=loaded_agent, 
    filename="logs/videos/reinforce_flight_loaded.mp4", 
    seconds=60
)


--- Recording 60s video for Custom REINFORCE ---


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (600, 400) to (608, 400) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Successfully saved video to: logs/videos/reinforce_flight_loaded.mp4
